# EX5 Bending Forward Reconstruction with RT-RPINN

- This notebook trains the forward RT-RPINN field model for the EX5 bending shape-memory cycle.
- The material parameters are fixed at their reference values; only the displacement-field network is trained.

Run the cells in order. The paths are kept consistent with the original EX5 Python scripts.


## Script Notes

Example 5 (FORWARD): Full-field reconstruction of the bending shape-memory cycle
with a Spatiotemporal PINN (MLP backbone), using KNOWN material parameters.

This is the forward counterpart to inverse_spatiotemporal_pinn_ex5.py. The shift
parameters (WLF C1/C2 + Arrhenius Ea/R) are held FIXED at their ground-truth
values and only the displacement field network is trained, from:
  - full-field displacement / strain / stress data
  - left-face clamped BC, right-face prescribed-rotation BC (until release at 70 s)
  - traction-free lateral / released faces
  - (optionally) PDE equilibrium

Observables: this case has NO reaction force. The boundary observables are the
reaction MOMENT (RM) and end ROTATION (UR); neither is used as a training target
here (RM supervision is still under consideration; UR is a rotation incompatible
with the displacement output). They remain validation-only quantities.

See inverse_spatiotemporal_pinn_ex5.py for the full physics / architecture.


In [ ]:
# Notebook path setup
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'EX-5-RESULTS').exists() and (NOTEBOOK_DIR / 'EX5' / 'EX-5-RESULTS').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'EX5'
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
print(f'EX5 notebook directory: {NOTEBOOK_DIR}')


## Imports and definitions


In [ ]:
import sys
import argparse
import torch
from pathlib import Path

from inverse_spatiotemporal_pinn_ex5 import (
    _Tee,
    device,
    FEDataLoader,
    InverseShiftParams,
    SpatiotemporalPINN,
    InversePINNSolver,
    plot_training_history,
    set_global_seed,
)


def main():
    parser = argparse.ArgumentParser(
        description="EX5 bending FORWARD field PINN (MLP) with fixed (true) material parameters."
    )
    parser.add_argument('--epochs', type=int, default=10000,
                        help='Training epochs.')
    parser.add_argument('--batch-size', type=int, default=256,
                        help='Spatial batch size per sampled time step.')
    parser.add_argument('--seq-length', type=int, default=80,
                        help='Temporal sequence length sampled each epoch.')
    parser.add_argument('--lr-network', type=float, default=2e-3,
                        help='Learning rate for field network.')
    parser.add_argument('--adaptive-loss-balance', dest='adaptive_loss_balance', action='store_true', default=True,
                        help='Enable uncertainty-based adaptive loss balancing.')
    parser.add_argument('--no-adaptive-loss-balance', dest='adaptive_loss_balance', action='store_false',
                        help='Disable adaptive loss balancing.')
    parser.add_argument('--adaptive-lr', dest='adaptive_lr', action='store_true', default=True,
                        help='Enable plateau-triggered adaptive learning-rate reduction.')
    parser.add_argument('--no-adaptive-lr', dest='adaptive_lr', action='store_false',
                        help='Disable adaptive learning-rate logic.')
    parser.add_argument('--stride', type=int, default=5,
                        help='Frame stride for loading FE results (used when --n-temporal<=0).')
    parser.add_argument('--n-temporal', type=int, default=150,
                        help='Sparse temporal subsampling: target total frames to load '
                             'across the cycle (~1296 available). 0 or negative → use --stride.')
    parser.add_argument('--n-spatial', type=int, default=3000,
                        help='Sparse spatial subsampling: number of nodes to keep '
                             '(end faces kept in full). 0 or negative → keep all 14454.')
    parser.add_argument('--seed', type=int, default=42,
                        help='Global random seed.')
    args = parser.parse_args(args=[])
    set_global_seed(args.seed)
    n_spatial = args.n_spatial if args.n_spatial and args.n_spatial > 0 else None
    n_temporal = args.n_temporal if args.n_temporal and args.n_temporal > 0 else None

    print("=" * 70)
    print("EX5 FORWARD FIELD RECONSTRUCTION — SPATIOTEMPORAL PINN (MLP)")
    print("Bending shape-memory cycle (95x13x2 beam, end rotation UR=4.71 rad)")
    print("Material parameters: FIXED at ground truth (forward problem)")
    print("=" * 70)
    print(f"Seed: {args.seed}")

    script_dir = NOTEBOOK_DIR
    data_dir   = script_dir / 'EX-5-RESULTS'
    rm_file    = script_dir / 'ex-5-Bending-RM.csv'   # reaction moment (validation only)
    ur_file    = script_dir / 'ex-5-Bending-UR.csv'   # end rotation (validation only)
    ft_file    = script_dir / 'frame-time.csv'

    print("\nLoading FE data...")
    fe_loader = FEDataLoader(data_dir, rm_file, ur_file, ft_file, stride=args.stride,
                             n_spatial=n_spatial, spatial_seed=args.seed,
                             n_temporal=n_temporal)
    fe_loader.load_all_data()
    bounds = fe_loader.get_domain_bounds()

    print("\nDomain bounds:")
    for k, v in bounds.items():
        print(f"  {k}: {v:.2f}")


## Characteristic displacement scale for output scaling (large-deformation bending).


In [ ]:
import numpy as np
    u_mag = np.linalg.norm(
        fe_loader.full_data[['U1', 'U2', 'U3']].values.astype(float), axis=1)
    u_scale = float(1.1 * np.max(u_mag))
    print(f"\nOutput displacement scale: {u_scale:.2f} mm (1.1 x max|u|)")


## Shift parameters fixed at ground truth.


In [ ]:
mat_params = InverseShiftParams().to(device)
    mat_params.set_to_true()
    print("\nFixed (true) shift parameters:")
    print(f"  C1   = {mat_params.C1.item():.4f}")
    print(f"  C2   = {mat_params.C2.item():.4f} K")
    print(f"  Ea/R = {mat_params.Ea_R.item():.2f} K")

    model = SpatiotemporalPINN(hidden=(128, 128, 128, 128),
                               output_scale=u_scale).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\nPINN parameters: {n_params:,}")


## therefore fit u DIRECTLY and dominantly, dropping strain/stress/traction.


In [ ]:
solver = InversePINNSolver(
        model, mat_params, fe_loader, bounds,
        lambda_data=20.0,    # ← dominant: direct supervision on the displacement field
        lambda_strain=0.0,   # small-strain measure invalid under large rotation
        lambda_stress=0.0,   # small-strain stress invalid under large rotation
        lambda_pde=0.0,
        lambda_bc_left=1.0,  # light clamp emphasis (data already covers left face)
        lambda_bc_right=0.0, # prescribed/released end already in the data
        lambda_traction=0.0, # small-strain traction invalid under large rotation
        lambda_rf=0.0,
        lambda_obs=0.0,
        lambda_obs_rate=0.0,
        freeze_shift_params=True,   # ← forward mode: material parameters held fixed
    )

    solver.train(
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr_network=args.lr_network,
        lr_params=0.0,
        log_interval=100,
        seq_length=args.seq_length,
        adaptive_loss_balance=args.adaptive_loss_balance,
        adaptive_lr=args.adaptive_lr,
    )


## Save results


In [ ]:
print("\nSaving results...")
    solver.save_loss_history(script_dir / 'ex5_pinn_forward_loss_history.csv')
    solver.save_param_history(script_dir / 'ex5_pinn_forward_param_history.csv')
    plot_training_history(solver, script_dir,
                          filename='ex5_pinn_forward_training_history.png')

    model_path = script_dir / 'ex5_pinn_forward_model.pth'
    torch.save({
        'model_state_dict':      model.state_dict(),
        'mat_params_state_dict': mat_params.state_dict(),
        'fixed_params': {
            'C1':   mat_params.C1.item(),
            'C2':   mat_params.C2.item(),
            'Ea_R': mat_params.Ea_R.item(),
        },
    }, model_path)
    print(f"Model saved to {model_path}")

    print("\n" + "=" * 70)
    print("EX5 FORWARD RECONSTRUCTION COMPLETE")
    print("=" * 70)


if __name__ == "__main__":
    _script_dir = NOTEBOOK_DIR
    _log_path = _script_dir / 'ex5_pinn_forward_training_log.txt'
    _tee = _Tee(sys.stdout, _log_path)
    sys.stdout = _tee
    try:
        main()
    finally:
        sys.stdout = _tee._orig
        _tee.close()
        _tee._orig.write(f"\nLog saved to: {_log_path}\n")
